In [21]:
import pandas as pd

In [22]:
# Load mandate start dates
mandate_df = pd.read_csv("../data/mandate_start_dates.csv")

mandate_df["Date"] = pd.to_datetime(mandate_df["Date"])

# Create a dictionary: {state: mandate_start_date}
state_mandate_dict = dict(zip(mandate_df["RegionName"], mandate_df["Date"]))

state_mandate_dict

{'Australian Capital Territory': Timestamp('2021-08-18 00:00:00'),
 'New South Wales': Timestamp('2021-07-09 00:00:00'),
 'Northern Territory': Timestamp('2021-11-21 00:00:00'),
 'Queensland': Timestamp('2021-01-17 00:00:00'),
 'South Australia': Timestamp('2021-07-26 00:00:00'),
 'Tasmania': Timestamp('2021-10-21 00:00:00'),
 'Victoria': Timestamp('2020-07-21 00:00:00'),
 'Western Australia': Timestamp('2021-02-08 00:00:00')}

In [23]:
# Read data
YouGov = pd.read_csv("../data/cleaned_YouGov.csv")

# Convert time to datetime
YouGov["endtime"] = pd.to_datetime(YouGov["endtime"])

YouGov["within_mandate_period"] = 0

state_dummy_cols = [col for col in YouGov.columns if col.startswith("state_")]

# Loop through each state and its mandate start date
for state, start_date in state_mandate_dict.items():
    state_col = f"state_{state}"
    
    # Select rows belonging to this state and also after mandate start
    if state_col in YouGov.columns:
        mask = (YouGov[state_col] == 1) & (YouGov["endtime"] >= start_date)
    else:
         # This state is the reference state omitted by dummy encoding
        mask = (YouGov[state_dummy_cols].sum(axis=1) == 0) & (YouGov["endtime"] >= start_date)
    
    YouGov.loc[mask, "within_mandate_period"] = 1

In [24]:
# Split datasets
before_mandate_df = YouGov[YouGov["within_mandate_period"] == 0].copy()
after_mandate_df = YouGov[YouGov["within_mandate_period"] == 1].copy()

before_mandate_df = before_mandate_df.drop(columns=["within_mandate_period", "endtime"])
after_mandate_df  = after_mandate_df .drop(columns=["within_mandate_period", "endtime"])

before_mandate_df.shape, after_mandate_df.shape

((14842, 59), (25048, 59))

In [25]:
before_mandate_df.to_csv("../data/before_mandate_data.csv", index=False)
after_mandate_df.to_csv("../data/after_mandate_data.csv", index=False)